In [31]:

from datasets import load_dataset


#loading from hugging face
dataset = load_dataset("tattabio/ec_classification")
## making test and train set
train_df = dataset["train"].to_pandas()
test_df = dataset["test"].to_pandas()
##sanity check to make sure dims are the same as on the DGEB paper dimensions
print(train_df.shape)
print(test_df.shape)

(512, 3)
(128, 3)


In [32]:
train_df.head()
## checking what the label and seq loomks like 

,Entry,Label,Sequence
0,Q9LQC0,1.14.14.18,MATSRLNASCRFPASRRLDCESYVSLRAKTVTIRYVRTIAAPRRHL...
1,A0AFT8,1.14.14.18,MIIVTNTIKVEKGAAEHVIRQFTGANGDGHPTKDIAEVEGFLGFEL...
2,P74133,1.14.14.18,MTNLAQKLRYGTQQSHTLAENTAYMKCFLKGIVEREPFRQLLANLY...
3,O31534,1.14.14.18,MFVQLRKMTVKEGFADKVIERFSAEGIIEKQEGLIDVTVLEKNVRR...
4,P34697,1.15.1.1,MFMNLLTQVSNAIFPQVEAAQKMSNRAVAVLRGETVTGTIWITQKS...


In [33]:
x_train_seq = train_df["Sequence"].tolist()
y_train_labels = train_df["Label"].tolist()

x_test_seq = test_df["Sequence"].tolist()
y_test_labels = test_df["Label"].tolist()

from sklearn.linear_model import LogisticRegression
## defining model
model =LogisticRegression(max_iter=1000)




In [ ]:

import dgeb
import os

os.environ["OMP_NUM_THREADS"] = "1"
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
## first input which is just the embeddings i am using dgeb embedding model to make each seq into a vector
embedding_model = dgeb.get_model("facebook/esm2_t6_8M_UR50D")



In [28]:
## converting train data to numerical vectors
x_train_input1 = embedding_model.encode(x_train_seq)[:, -1, :]

/opt/anaconda3/envs/knnlab/lib/python3.10/site-packages/torch/utils/data/dataloader.py:627: UserWarning: This DataLoader will create 16 worker processes in total. Our suggested max number of worker in current system is 8 (`cpuset` is not taken into account), which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(
encoding:   0%|          | 0/4 [00:00<?, ?it/s]/opt/anaconda3/envs/knnlab/lib/python3.10/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)
encoding: 100%|██████████| 4/4 [04:15<00:00, 63.89s/it]


In [ ]:
## doing same for test data
## broke into two cells so we would avoid rerunning

import numpy as np
x_test_p1 = embedding_model.encode(x_test_seq[:127])[:, -1, :]
last_test = embedding_model.encode(x_test_seq[127:])[:, -1, :]
x_test_input1 = np.vstack([x_test_p1, last_test])


In [30]:
from sklearn.metrics import accuracy_score, f1_score 
## checking shape
print(x_train_input1.shape)
print(x_test_input1.shape)
## now fitting model on this info
model.fit(x_train_input1, y_train_labels)
predictions_input1 = model.predict(x_test_input1)

## accuracy and f1 score for input 1
print("accuracy:", accuracy_score(y_test_labels, predictions_input1))
print("f1:", f1_score(y_test_labels, predictions_input1, average="macro"))

(512, 320)
(128, 320)
accuracy: 0.4921875
f1: 0.43671875


In [25]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics import accuracy_score, f1_score 
## now doing input 2 which is k mer counts of 2 and 3

def kmerFunction(seq, k):
    return " ".join([seq[i:i+k] for i in range(len(seq) - k + 1)])


train_kmer2 = [kmerFunction(seq, 2) for seq in x_train_seq]
test_kmer2 = [kmerFunction(seq, 2) for seq in x_test_seq]

train_kmer3 = [kmerFunction(seq, 3) for seq in x_train_seq]
test_kmer3 = [kmerFunction(seq, 3) for seq in x_test_seq]

## for kmer 2
helper = CountVectorizer()
x_train_input2_kmer2 = helper.fit_transform(train_kmer2)
x_test_input2_kmer2 = helper.transform(test_kmer2)

## defining model again
model2kmer2 = LogisticRegression(max_iter=1000)
model2kmer2.fit(x_train_input2_kmer2, y_train_labels)
predictions_input2kmer2 = model2kmer2.predict(x_test_input2_kmer2)

## printing accuracy and f1 score
print("accuracy for kmer2:", accuracy_score(y_test_labels, predictions_input2kmer2))
print("f1 for kmer2:", f1_score(y_test_labels, predictions_input2kmer2, average="macro"))

## for kmer 3
helper = CountVectorizer()
x_train_input2_kmer3 = helper.fit_transform(train_kmer3)
x_test_input2_kmer3 = helper.transform(test_kmer3)

## defining model again
model2kmer3 = LogisticRegression(max_iter=1000)
model2kmer3.fit(x_train_input2_kmer3, y_train_labels)
predictions_input2kmer3 = model2kmer3.predict(x_test_input2_kmer3)

## printing accuracy and f1 score
print("accuracy for kmer3:", accuracy_score(y_test_labels, predictions_input2kmer3))
print("f1 for kmer3:", f1_score(y_test_labels, predictions_input2kmer3, average="macro"))

accuracy for kmer2: 0.25
f1 for kmer2: 0.2028645833333333
accuracy for kmer3: 0.2109375
f1 for kmer3: 0.1513206845238095


In [24]:
## now doing input 3 which is weighted kmer
from sklearn.feature_extraction.text import TfidfVectorizer

## defining the weighted helper
weighted2 =  TfidfVectorizer()
weighted3 =  TfidfVectorizer()

x_train_input3_kmer2 = weighted2.fit_transform(train_kmer2)
x_test_input3_kmer2 = weighted2.transform(test_kmer2)

x_train_input3_kmer3 = weighted3.fit_transform(train_kmer3)
x_test_input3_kmer3 = weighted3.transform(test_kmer3)

## making model3 for kmer2
model3_kmer2 = LogisticRegression(max_iter=1000)
model3_kmer2.fit(x_train_input3_kmer2, y_train_labels)
predictions_input3_kmer2 = model3_kmer2.predict(x_test_input3_kmer2)
## f1 and accuracy scores
print("accuracy for kmer2:", accuracy_score(y_test_labels, predictions_input3_kmer2))
print("f1 for kmer2:", f1_score(y_test_labels, predictions_input3_kmer2, average="macro"))

## making model3 for kmer3
model3_kmer3 = LogisticRegression(max_iter=1000)
model3_kmer3.fit(x_train_input3_kmer3, y_train_labels)
predictions_input3_kmer3 = model3_kmer3.predict(x_test_input3_kmer3)
## f1 and accuracy scores
print("accuracy for kmer3:", accuracy_score(y_test_labels, predictions_input3_kmer3))
print("f1 for kmer3:", f1_score(y_test_labels, predictions_input3_kmer3, average="macro"))

accuracy for kmer2: 0.2109375
f1 for kmer2: 0.15260416666666668
accuracy for kmer3: 0.25
f1 for kmer3: 0.20703124999999997
